<a href="https://colab.research.google.com/github/yuashi/AI_Shopping_Agent/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing Dependencies

In [ ]:
!pip install -q -U transformers==4.46.3 datasets==3.1.0 accelerate==1.1.1 peft==0.13.2 trl==0.12.1 "bitsandbytes>=0.45.2" huggingface_hub==0.26.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.9/310.9 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 91.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires 

In [ ]:
import transformers, datasets, accelerate, peft, trl, bitsandbytes, huggingface_hub
print(transformers.__version__, datasets.__version__, peft.__version__, trl.__version__)

4.46.3 3.1.0 0.13.2 0.12.1


Hugging Face Authentication

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

Loading and Splitting the Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("b-mc2/sql-create-context", split="train")
dataset = dataset.shuffle(seed=42).select(range(20000))
dataset = dataset.train_test_split(test_size=0.05, seed=42)
print(dataset)
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['answer', 'question', 'context'],
        num_rows: 19000
    })
    test: Dataset({
        features: ['answer', 'question', 'context'],
        num_rows: 1000
    })
})
{'answer': 'SELECT method FROM table_name_42 WHERE event = "flawless fighting championship 1: the beginning"', 'question': "Which method's event was flawless fighting championship 1: the beginning?", 'context': 'CREATE TABLE table_name_42 (method VARCHAR, event VARCHAR)'}


Prompt Formatting

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SYSTEM_PROMPT = (
    "You are a SQL expert. Given a database schema and a question, write the "
    "correct SQL query. Respond with only the SQL query, no explanation."
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def format_example(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Schema:\n{example['context']}\n\nQuestion: {example['question']}"},
        {"role": "assistant", "content": example["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}


dataset = dataset.map(format_example)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/19000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading base model in 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LoRA Configuration

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

Training

In [ ]:
from trl import SFTTrainer, SFTConfig

ADAPTER_DIR = "qwen2.5-1.5b-text2sql-lora"

sft_config = SFTConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    save_strategy="epoch",
    bf16=True,
    max_seq_length=512,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=lora_config,
)
trainer.train()
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

Map:   0%|          | 0/19000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,1.180700
40,0.723000
60,0.646400
80,0.600400
100,0.610200
120,0.621500
140,0.626200
160,0.613800
180,0.601100
200,0.582200


('qwen2.5-1.5b-text2sql-lora/tokenizer_config.json',
 'qwen2.5-1.5b-text2sql-lora/special_tokens_map.json',
 'qwen2.5-1.5b-text2sql-lora/vocab.json',
 'qwen2.5-1.5b-text2sql-lora/merges.txt',
 'qwen2.5-1.5b-text2sql-lora/added_tokens.json',
 'qwen2.5-1.5b-text2sql-lora/tokenizer.json')

Evaluation (exact-match + execution validity)

In [ ]:
import re
import sqlite3
from peft import PeftModel


def normalize_sql(sql: str) -> str:
    sql = sql.strip().rstrip(";").lower()
    return re.sub(r"\s+", " ", sql)


def is_valid_sql(schema: str, sql: str) -> bool:
    try:
        conn = sqlite3.connect(":memory:")
        cur = conn.cursor()
        cur.executescript(schema)
        cur.execute(sql)
        conn.close()
        return True
    except Exception:
        return False


def generate_sql(model, tokenizer, schema, question) -> str:
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Schema:\n{schema}\n\nQuestion: {question}"},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    # LoRA adapter weights stay float32 by design even though the base model
    # runs in bf16/4-bit. SFTTrainer handled that mismatch via an implicit
    # autocast context during training; generate() has none by default, so
    # without this wrap you'd hit "expected scalar type Float but found BFloat16".
    with torch.autocast(device_type=model.device.type, dtype=torch.bfloat16):
        out = model.generate(**inputs, max_new_tokens=120, temperature=0.1, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


# `model` already has the LoRA adapter injected in-place by SFTTrainer during
# training -- reloading it via PeftModel.from_pretrained(model, ADAPTER_DIR)
# double-wraps it and reloads the adapter weights in float32, which clashes
# with the bf16/4-bit base model and raises
# "RuntimeError: expected scalar type Float but found BFloat16". Just eval-mode
# the model you already have.
eval_model = model
eval_model.eval()

EVAL_SAMPLE_SIZE = 300  # subsample of the held-out test split, keeps this within a Colab session
eval_set = dataset["test"].shuffle(seed=42).select(range(min(EVAL_SAMPLE_SIZE, len(dataset["test"]))))

exact_matches = 0
valid_executions = 0
gold_valid = 0

for ex in eval_set:
    generated = generate_sql(eval_model, tokenizer, ex["context"], ex["question"])
    if normalize_sql(generated) == normalize_sql(ex["answer"]):
        exact_matches += 1
    if is_valid_sql(ex["context"], generated):
        valid_executions += 1
    if is_valid_sql(ex["context"], ex["answer"]):
        gold_valid += 1

n = len(eval_set)
print(f"Evaluated on {n} held-out examples")
print(f"Exact-match accuracy:    {exact_matches / n:.1%}")
print(f"Execution validity rate: {valid_executions / n:.1%}  (gold queries: {gold_valid / n:.1%} valid, as a sanity ceiling)")

Evaluated on 300 held-out examples
Exact-match accuracy:    73.0%
Execution validity rate: 96.7%  (gold queries: 97.7% valid, as a sanity ceiling)


Merging Adapter into Base model

In [ ]:
MERGED_DIR = "qwen2.5-1.5b-text2sql-merged"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)
merged = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged = merged.merge_and_unload()
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

('qwen2.5-1.5b-text2sql-merged/tokenizer_config.json',
 'qwen2.5-1.5b-text2sql-merged/special_tokens_map.json',
 'qwen2.5-1.5b-text2sql-merged/vocab.json',
 'qwen2.5-1.5b-text2sql-merged/merges.txt',
 'qwen2.5-1.5b-text2sql-merged/added_tokens.json',
 'qwen2.5-1.5b-text2sql-merged/tokenizer.json')

In [ ]:
from huggingface_hub import HfApi, create_repo

create_repo("yuashi/qwen2.5-1.5b-text2sql-merged", exist_ok=True)
HfApi().upload_folder(
    folder_path="qwen2.5-1.5b-text2sql-merged",
    repo_id="yuashi/qwen2.5-1.5b-text2sql-merged",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...sql-merged/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...-merged/model.safetensors:   1%|1         | 40.0MB / 3.09GB            

CommitInfo(commit_url='https://huggingface.co/yuashi/qwen2.5-1.5b-text2sql-merged/commit/e28f2a24b3b6daad818b10820ca6d0115e6d7b66', commit_message='Upload folder using huggingface_hub', commit_description='', oid='e28f2a24b3b6daad818b10820ca6d0115e6d7b66', pr_url=None, repo_url=RepoUrl('https://huggingface.co/yuashi/qwen2.5-1.5b-text2sql-merged', endpoint='https://huggingface.co', repo_type='model', repo_id='yuashi/qwen2.5-1.5b-text2sql-merged'), pr_revision=None, pr_num=None)